# Member Experience KPI Dashboard

Comprehensive analytics dashboard tracking healthcare member experience across:
- Satisfaction & NPS trends
- Call center performance (FCR, handle time, abandonment)
- Digital engagement & channel adoption
- Operational turnaround times
- Regional benchmarking
- Retention & churn analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.kpi_engine import (
    compute_csat, compute_nps, compute_fcr, compute_avg_handle_time,
    compute_abandonment_rate, compute_portal_adoption, compute_mobile_adoption,
    compute_online_claim_rate, compute_avg_claim_processing, compute_avg_prior_auth,
    compute_disenrollment_rate, compute_voluntary_churn_rate,
    monthly_kpi_summary, regional_kpi_summary
)
from src.visualizations import (
    plot_kpi_tiles, plot_monthly_trends, plot_regional_comparison,
    plot_channel_mix, plot_satisfaction_distribution
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/member_experience_data.csv')
disenroll_df = pd.read_csv('../data/disenrollment_data.csv')
members_df = pd.read_csv('../data/members.csv')

print(f'Interactions: {len(df):,}')
print(f'Unique members: {df["member_id"].nunique():,}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Regions: {df["region"].nunique()}')
print(f'Disenrollments: {len(disenroll_df)}')
df.head(3)

## 2. Executive KPI Summary

In [ ]:
total_members = len(members_df)
csat = compute_csat(df)
nps = compute_nps(df)
fcr = compute_fcr(df)
aht = compute_avg_handle_time(df)
abandon = compute_abandonment_rate(df)
portal = compute_portal_adoption(df)
mobile = compute_mobile_adoption(df)
disenroll_rate = compute_disenrollment_rate(disenroll_df, total_members)
vol_churn = compute_voluntary_churn_rate(disenroll_df, total_members)
claim_days = compute_avg_claim_processing(df)
pa_days = compute_avg_prior_auth(df)

kpis = {
    'CSAT Score': (f'{csat:.1f}/10', '#3498DB'),
    'NPS': (f'{nps:+.0f}', '#27AE60' if nps > 0 else '#E74C3C'),
    'FCR Rate': (f'{fcr:.0%}', '#27AE60' if fcr > 0.75 else '#F39C12'),
    'Avg Handle Time': (f'{aht:.1f} min', '#3498DB'),
    'Abandonment Rate': (f'{abandon:.1%}', '#E74C3C' if abandon > 0.10 else '#27AE60'),
    'Portal Adoption': (f'{portal:.0%}', '#27AE60' if portal > 0.50 else '#F39C12'),
    'Mobile Adoption': (f'{mobile:.0%}', '#27AE60' if mobile > 0.30 else '#F39C12'),
    'Claim Processing': (f'{claim_days:.0f} days', '#F39C12' if claim_days > 10 else '#27AE60'),
    'Prior Auth TAT': (f'{pa_days:.0f} days', '#E74C3C' if pa_days > 5 else '#27AE60'),
    'Disenrollment': (f'{disenroll_rate:.1%}', '#E74C3C' if disenroll_rate > 0.10 else '#F39C12'),
    'Voluntary Churn': (f'{vol_churn:.1%}', '#E74C3C' if vol_churn > 0.05 else '#F39C12'),
    'Total Interactions': (f'{len(df):,}', '#3498DB'),
}

fig = plot_kpi_tiles(kpis, title='Healthcare Member Experience — Executive KPI Dashboard')
plt.show()

## 3. Satisfaction & NPS Analysis

In [ ]:
fig = plot_satisfaction_distribution(df)
plt.show()

print('\nSatisfaction by plan type:')
print(df.groupby('plan_type')['satisfaction_score'].agg(['mean', 'median', 'std']).round(2))

## 4. Monthly KPI Trends

In [ ]:
monthly = monthly_kpi_summary(df)

metrics = ['csat', 'nps', 'fcr', 'avg_handle_time', 'abandonment_rate', 'portal_adoption']
titles = ['CSAT Score', 'Net Promoter Score', 'First Contact Resolution',
          'Avg Handle Time (min)', 'Abandonment Rate', 'Portal Adoption Rate']

fig = plot_monthly_trends(monthly, metrics, titles, figsize=(16, 12))
plt.show()

print('\nMonthly KPI Table:')
monthly.round(2)

## 5. Regional Benchmarking

In [ ]:
regional = regional_kpi_summary(df)

fig = plot_regional_comparison(regional, 'csat', 'CSAT Score by Region')
plt.show()

fig = plot_regional_comparison(regional, 'fcr', 'FCR Rate by Region')
plt.show()

fig = plot_regional_comparison(regional, 'nps', 'NPS by Region')
plt.show()

print('\nFull Regional KPI Comparison:')
regional.set_index('region').round(3)

## 6. Channel Mix & Digital Engagement

In [ ]:
fig = plot_channel_mix(df)
plt.show()

online_rate = compute_online_claim_rate(df)
print(f'\nOnline claim submission rate: {online_rate:.1%}')
print(f'Portal adoption: {compute_portal_adoption(df):.1%}')
print(f'Mobile app adoption: {compute_mobile_adoption(df):.1%}')

print('\nDigital adoption by age group:')
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], labels=['18-30', '31-45', '46-60', '60+'])
print(df.groupby('age_group')[['used_portal', 'used_mobile_app']].mean().round(3))

## 7. Operational Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Claim processing time distribution
claims_data = df.dropna(subset=['claim_processing_days'])
axes[0].hist(claims_data['claim_processing_days'], bins=20, edgecolor='black', alpha=0.7, color='#3498DB')
axes[0].axvline(claims_data['claim_processing_days'].mean(), color='red', linestyle='--',
                label=f'Mean: {claims_data["claim_processing_days"].mean():.1f} days')
axes[0].set_title('Claim Processing Time Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Days')
axes[0].legend()

# Prior auth turnaround
pa_data = df.dropna(subset=['prior_auth_days'])
axes[1].hist(pa_data['prior_auth_days'], bins=15, edgecolor='black', alpha=0.7, color='#F39C12')
axes[1].axvline(pa_data['prior_auth_days'].mean(), color='red', linestyle='--',
                label=f'Mean: {pa_data["prior_auth_days"].mean():.1f} days')
axes[1].set_title('Prior Auth Turnaround Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Days')
axes[1].legend()

# Handle time by reason
phone_data = df[df['channel'] == 'phone'].dropna(subset=['handle_time_minutes'])
reason_ht = phone_data.groupby('contact_reason')['handle_time_minutes'].mean().sort_values()
axes[2].barh(reason_ht.index, reason_ht.values, color='#4C72B0', edgecolor='black')
axes[2].set_title('Avg Handle Time by Contact Reason', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Minutes')
for i, v in enumerate(reason_ht.values):
    axes[2].text(v + 0.05, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.show()

## 8. Retention & Churn Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Disenrollment by reason
reason_counts = disenroll_df['disenroll_reason'].value_counts()
axes[0].pie(reason_counts.values, labels=reason_counts.index, autopct='%1.0f%%',
            colors=sns.color_palette('muted', len(reason_counts)), startangle=90)
axes[0].set_title('Disenrollment Reasons', fontsize=13, fontweight='bold')

# Disenrollment by region
region_disenroll = disenroll_df.groupby('region').size()
region_total = members_df['region'].value_counts()
region_rate = (region_disenroll / region_total).sort_values()
colors = ['#E74C3C' if v > region_rate.mean() else '#27AE60' for v in region_rate.values]
axes[1].barh(region_rate.index, region_rate.values, color=colors, edgecolor='black')
axes[1].set_title('Disenrollment Rate by Region', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Rate')
axes[1].axvline(region_rate.mean(), color='gray', linestyle='--', alpha=0.7)
for i, v in enumerate(region_rate.values):
    axes[1].text(v + 0.001, i, f'{v:.1%}', va='center')

# Monthly disenrollment trend
disenroll_df['month'] = pd.to_datetime(disenroll_df['disenroll_date']).dt.to_period('M').astype(str)
monthly_disenroll = disenroll_df.groupby('month').size()
axes[2].plot(monthly_disenroll.index, monthly_disenroll.values, marker='o', linewidth=2, color='#E74C3C')
axes[2].fill_between(range(len(monthly_disenroll)), monthly_disenroll.values, alpha=0.1, color='#E74C3C')
axes[2].set_title('Monthly Disenrollments', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f'Total disenrollments: {len(disenroll_df)}')
print(f'Disenrollment rate: {len(disenroll_df)/len(members_df):.1%}')
print(f'Voluntary churn rate: {disenroll_df["voluntary"].mean():.1%} of disenrollments')

## 9. Key Findings & Recommendations

### Findings
1. **Grievance and billing contacts drive lowest satisfaction** — targeted intervention on these categories would move overall CSAT
2. **Digital adoption varies significantly by age** — younger members embrace self-service; older members depend on phone
3. **Regional performance gaps exist** — identify best practices from top-performing regions and replicate
4. **Claim processing times show high variance** — standardizing workflows could reduce outliers and improve satisfaction
5. **Voluntary churn concentrated in cost and service quality** — pricing and member experience are the levers

### Recommendations
1. **Prioritize grievance resolution training** — coaching on de-escalation and first-contact resolution
2. **Expand digital self-service** — invest in portal/mobile features for claims tracking and billing disputes
3. **Implement regional scorecards** — monthly benchmarking with action plans for underperforming regions
4. **Automate prior auth workflows** — reduce turnaround time to under 3 days
5. **Launch retention outreach** — proactive engagement for members showing disengagement signals